# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col="expr"` or `col=callable`). Each entry is evaluated in order via pandas `eval()` — a plain formula per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments** (most Pythonic): `.pt.mutate(bmi="body_mass_g / bill_length_mm ** 2", mass_kg="body_mass_g / 1000")`
2. **Dictionary unpacking** (for new column names with spaces): `.pt.mutate(**{"body mass kg": "body_mass_g / 1000"})`
3. **External file specification**: `.pt.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via kwargs
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,body_mass_g,bill_length_mm,bmi
240,Gentoo,5700.0,50.2,2.261869
291,Gentoo,5000.0,46.4,2.322384
7,Adelie,4675.0,39.2,3.042352
92,Adelie,3400.0,34.0,2.941176
91,Adelie,4300.0,41.1,2.545569
147,Adelie,3475.0,36.6,2.594135
165,Chinstrap,4050.0,52.0,1.497781
35,Adelie,4150.0,39.2,2.700698
4,Adelie,3450.0,36.7,2.561456
206,Chinstrap,3350.0,42.5,1.854671


In [3]:
# Column names inside the expression must stay unquoted — quoting one turns it into
# a string literal, which will break arithmetic
try:
    penguins.pt.mutate(bmi="'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .pt.mutate(heavy='body_mass_g > 4000', mass_kg='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'heavy')
    .sample(10)
)


,species,body_mass_g,mass_kg,heavy
104,Adelie,2925.0,2.925,False
20,Adelie,3400.0,3.400,False
60,Adelie,3150.0,3.150,False
268,Gentoo,5100.0,5.100,True
4,Adelie,3450.0,3.450,False
233,Gentoo,5850.0,5.850,True
182,Chinstrap,3200.0,3.200,False
218,Chinstrap,4100.0,4.100,True
128,Adelie,3050.0,3.050,False
329,Gentoo,5500.0,5.500,True


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .pt.mutate(mass_kg='body_mass_g / 1000', mass_lb='mass_kg * 2.20462')
    .pt.select('species', 'body_mass_g', 'mass_kg', 'mass_lb')
    .sample(10)
)


,species,body_mass_g,mass_kg,mass_lb
195,Chinstrap,3500.0,3.500,7.716170
212,Chinstrap,3950.0,3.950,8.708249
221,Gentoo,5700.0,5.700,12.566334
18,Adelie,3325.0,3.325,7.330361
333,Gentoo,5500.0,5.500,12.125410
218,Chinstrap,4100.0,4.100,9.038942
64,Adelie,2850.0,2.850,6.283167
252,Gentoo,5050.0,5.050,11.133331
230,Gentoo,4650.0,4.650,10.251483
154,Chinstrap,3650.0,3.650,8.046863


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# String comparisons inside the expression formula
(
    penguins
    .pt.mutate(is_adelie="species == 'Adelie'")
    .pt.select('species', 'is_adelie')
    .sample(10)
)


,species,is_adelie
158,Chinstrap,False
81,Adelie,True
328,Gentoo,False
157,Chinstrap,False
173,Chinstrap,False
26,Adelie,True
29,Adelie,True
106,Adelie,True
131,Adelie,True
293,Gentoo,False


In [7]:
# @-prefixed names resolve against the scope that called mutate(), not the module
# — works identically whether calling pt.mutate() or chaining .pt.mutate()
threshold = 4000

(
    penguins
    .pt.mutate(heavy='body_mass_g >= @threshold')
    .pt.select('species', 'body_mass_g', 'heavy')
    .sample(10)
)


,species,body_mass_g,heavy
31,Adelie,3900.0,False
284,Gentoo,4700.0,True
139,Adelie,4250.0,True
120,Adelie,3150.0,False
205,Chinstrap,4050.0,True
191,Chinstrap,4500.0,True
103,Adelie,4250.0,True
262,Gentoo,4300.0,True
170,Chinstrap,3450.0,False
178,Chinstrap,3400.0,False


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .pt.mutate(body_mass_g='body_mass_g / 1000')
    .pt.select('species', 'body_mass_g')
    .sample(10)
)


,species,body_mass_g
201,Chinstrap,3.675
67,Adelie,4.100
153,Chinstrap,3.900
22,Adelie,3.800
107,Adelie,3.900
89,Adelie,3.600
175,Chinstrap,3.800
310,Gentoo,4.950
292,Gentoo,5.100
57,Adelie,3.800


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [9]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3750, 4200], 'bill length mm': [39.1, 46.5]})

# Backticks protect column names that contain spaces
spaced.pt.mutate(bmi="`body mass g` / `bill length mm` ** 2")


,body mass g,bill length mm,bmi
0,3750,39.1,2.452888
1,4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [10]:
(
    penguins
    .pt.mutate(bmi='body_mass_g / bill_length_mm ** 2')
    .pt.qry(bmi='> 2')
    .pt.select('species', 'island', 'body_mass_g', 'bill_length_mm', 'bmi')
    .sample(10)
)


,species,island,body_mass_g,bill_length_mm,bmi
77,Adelie,Torgersen,3900.0,37.2,2.818245
92,Adelie,Dream,3400.0,34.0,2.941176
31,Adelie,Dream,3900.0,37.2,2.818245
14,Adelie,Torgersen,4400.0,34.6,3.675365
114,Adelie,Biscoe,3900.0,39.6,2.486991
247,Gentoo,Biscoe,5650.0,47.8,2.472821
333,Gentoo,Biscoe,5500.0,51.5,2.073711
74,Adelie,Torgersen,3700.0,35.5,2.935925
10,Adelie,Torgersen,3300.0,37.8,2.309566
60,Adelie,Biscoe,3150.0,35.7,2.471577


## Conditional column creation — `if_else()`, `case_when()`, `map()`
Plain `eval()` has no ternary/`where()` support, so `mutate()` provides three dplyr-style helpers as ordinary function calls, evaluated via `np.where()`/`np.select()`/`Series.map()`: `if_else(condition, true_value, false_value)`, `case_when((cond1, val1), (cond2, val2), ..., default)`, and `map(column, {key: value, ...}, default)`. A trailing bare argument to `case_when` is the catch-all default (like SQL ELSE). Because they are ordinary calls, they compose and chain with each other and with any pandas method. String outcomes need quotes; conditions are vectorized — prefer `and`/`or`/`not` (the bitwise `&`/`|`/`~` also work).

In [11]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
# true_value and false_value can be scalars or column names; conditions are vectorized
(
    penguins
    .pt.mutate(weight_class="if_else(body_mass_g > 4000, 'heavy', 'light')")
    .pt.select('species', 'body_mass_g', 'weight_class')
    .sample(10)
)


,species,body_mass_g,weight_class
250,Gentoo,5250.0,heavy
215,Chinstrap,4000.0,light
305,Gentoo,5600.0,heavy
104,Adelie,2925.0,light
154,Chinstrap,3650.0,light
47,Adelie,2975.0,light
325,Gentoo,5500.0,heavy
1,Adelie,3800.0,light
269,Gentoo,5300.0,heavy
69,Adelie,4450.0,heavy


In [12]:
# case_when supports tuples or flat pairs with default=
(
    penguins
    .pt.mutate(
        size_class="case_when(body_mass_g >= 4500, 'large', body_mass_g >= 3500, 'medium', default='small')"
    )
    .pt.select('species', 'body_mass_g', 'size_class')
    .sample(10)
)


,species,body_mass_g,size_class
120,Adelie,3150.0,small
176,Chinstrap,3300.0,small
150,Adelie,3700.0,medium
37,Adelie,3550.0,medium
54,Adelie,2900.0,small
306,Gentoo,4600.0,large
255,Gentoo,5400.0,large
298,Gentoo,4750.0,large
26,Adelie,3550.0,medium
296,Gentoo,4600.0,large


In [13]:
# map(column, {key: value, ...}, default) — recode a column through a dictionary lookup
# unmapped keys become default if given, else NaN
(
    penguins
    .pt.mutate(
        island_code="map(island, {'Torgersen': 'TOR', 'Biscoe': 'BIS'}, 'OTH')"
    )
    .pt.select('species', 'island', 'island_code')
    .sample(10)
)


,species,island,island_code
335,Gentoo,Biscoe,BIS
167,Chinstrap,Dream,OTH
252,Gentoo,Biscoe,BIS
316,Gentoo,Biscoe,BIS
265,Gentoo,Biscoe,BIS
305,Gentoo,Biscoe,BIS
313,Gentoo,Biscoe,BIS
331,Gentoo,Biscoe,BIS
341,Gentoo,Biscoe,BIS
306,Gentoo,Biscoe,BIS


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pd.DataFrame({
    'mobile': [None, '555-1234', None],
    'home': ['555-5678', None, None],
    'work': [None, None, '555-9012'],
})

contacts.pt.mutate(preferred_contact="coalesce(mobile, home, work, 'N/A')")


,mobile,home,work,preferred_contact
0,NaN,555-5678,NaN,555-5678
1,555-1234,NaN,NaN,555-1234
2,NaN,NaN,555-9012,555-9012


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .pt.mutate(
        bmi='body_mass_g / bill_length_mm ** 2',
        mass_kg='body_mass_g / 1000',
        is_heavy=lambda df: df['body_mass_g'] > 4000,
    )
    .pt.select('species', 'bmi', 'mass_kg', 'is_heavy')
    .head(5)
)


,species,bmi,mass_kg,is_heavy
0,Adelie,2.452888,3.75,False
1,Adelie,2.435507,3.80,False
2,Adelie,2.001121,3.25,False
3,Adelie,NaN,NaN,False
4,Adelie,2.561456,3.45,False


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [16]:
# Write a small demo spec file with comments and assignment '=' syntax
spec_content = """# Engineering features for penguins
mass_kg = body_mass_g / 1000

# Convert kg to lbs
mass_lb = mass_kg * 2.20462
"""
with open("penguin_features.txt", "w") as f:
    f.write(spec_content)

# Load directly using @filename
result = penguins.pt.mutate("@penguin_features.txt")
out = (
    result
    .pt.select("species", "mass_kg", "mass_lb")
    .sample(5)
)

# Clean up demo file
import os
os.remove("penguin_features.txt")

out


,species,mass_kg,mass_lb
14,Adelie,4.400,9.700328
88,Adelie,3.950,8.708249
278,Gentoo,4.450,9.810559
119,Adelie,3.325,7.330361
241,Gentoo,5.000,11.023100


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [17]:
spaced.pt.mutate(ratio='[body mass g] / [bill length mm]')


,body mass g,bill length mm,ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


## Creating new columns with spaces — dictionary unpacking (`**{...}`)

Python keyword argument syntax requires valid identifier names (`key=value`), so bare names with spaces like `my col = "..."` cause a Python `SyntaxError`.

To create a new column whose name contains spaces, pass the column definition in a dictionary and unpack it using `**`:


In [18]:
# Create a new column with spaces using dictionary unpacking
(
    spaced
    .pt.mutate(**{'body mass ratio': '[body mass g] / [bill length mm]'})
)


,body mass g,bill length mm,body mass ratio
0,3750,39.1,95.907928
1,4200,46.5,90.322581


In [19]:
# Define multiple new columns with spaces at once
(
    penguins
    .pt.mutate(**{
        'body mass kg': 'body_mass_g / 1000',
        'bill ratio': 'bill_length_mm / bill_depth_mm',
    })
    .pt.select('species', 'body mass kg', 'bill ratio')
    .head()
)


,species,body mass kg,bill ratio
0,Adelie,3.75,2.090909
1,Adelie,3.80,2.270115
2,Adelie,3.25,2.238889
3,Adelie,NaN,NaN
4,Adelie,3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```
